# building a chatbot

1. we will design and implement llm powered chatbot. this chatbot will be able to have a conversation and remember previous interactions.
2. we will use language model to have a conversation. there are several othere related concepts that include:

* conversational rag  : enable a chatbot experience over an external source of data
* agents : build a chatbot take action 

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


In [ ]:
from langchain_groq import ChatGroq
model = ChatGroq(groq_api_key=groq_api_key, model="llama-3.3-70b-versatile")

#this is just a stateless language model — it does not remember anything unless we pass previous messages.

In [15]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="hey! im prachi and im aiml engineer, passionate about learinging new things, can you tell me about langchain? ")])


AIMessage(content="Nice to meet you, Prachi. As an AIML engineer, you'll likely find LangChain fascinating. LangChain is an open-source framework that enables developers to build applications powered by large language models (LLMs) like LLaMA, PaLM, or others. It provides a simple and unified API for interacting with these models, making it easier to integrate them into various projects.\n\nLangChain allows you to:\n\n1. **Chain models together**: Combine the strengths of multiple models to achieve more complex tasks, such as text classification, question-answering, or text generation.\n2. **Use models as APIs**: Expose models as RESTful APIs, making it easy to integrate them into web applications, mobile apps, or other services.\n3. **Manage model serving**: Handle model deployment, scaling, and monitoring, ensuring that your applications can efficiently utilize the models.\n4. **Develop custom agents**: Create custom agents that can interact with models, allowing you to build more so

In [16]:
from langchain_core.messages import AIMessage
model.invoke([HumanMessage(content="hey! im prachi and im aiml engineer, passionate about learinging new things, can you tell me about langchain?"),
                AIMessage(content="Nice to meet you, Prachi. As an AIML engineer, you'll likely find LangChain fascinating. LangChain is an open-source framework that enables developers to build applications powered by large language models (LLMs) like LLaMA, PaLM, or others."),
                HumanMessage(content="Hey whats my name and what do i do?")])      


AIMessage(content="Your name is Prachi, and you're an AIML (Artificial Intelligence and Machine Learning) engineer. You're also passionate about learning new things.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 137, 'total_tokens': 169, 'completion_time': 0.070500188, 'completion_tokens_details': None, 'prompt_time': 0.013049765, 'prompt_tokens_details': None, 'queue_time': 0.159442914, 'total_time': 0.083549953}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_4f6d808339', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019df936-4175-7602-ae0c-5d8be28d7f81-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 32, 'total_tokens': 169})

## message history

* we can use message history class to wrap our models and make it stateful.
* this will keep track of inputs and outputs of the model store them in some db
* future interactions will then load those messages and pass them into chain as part of the input

In [ ]:
from ast import Store

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

#creates memory for each session

with_message_history = RunnableWithMessageHistory(model,get_session_history)

In [21]:
config={"configurable":{"session_id": "chat1"}}


In [24]:
response = with_message_history.invoke([HumanMessage(content="hey! im prachi and im aiml engineer, passionate about learinging new things, can you tell me about langchain? ")], 
                            config=config)


In [26]:
response.content

"Nice to meet you, Prachi. As an AIML engineer, you're likely interested in staying up-to-date with the latest developments in the field. LangChain is a relatively new and exciting concept that has gained significant attention in the AI community.\n\n**What is LangChain?**\nLangChain is an open-source framework that enables developers to build applications using large language models (LLMs) more efficiently. It provides a set of tools and APIs to simplify the process of integrating LLMs into various projects.\n\n**Key Features:**\n\n1. **Modular architecture**: LangChain allows developers to easily swap out different LLMs, fine-tune models, or add custom components.\n2. **Simple API**: The framework offers a simple and intuitive API, making it easier to integrate LLMs into applications.\n3. **Support for multiple models**: LangChain supports a wide range of LLMs, including popular models like BERT, RoBERTa, and transformer-based architectures.\n4. **Optimized performance**: The framewo

In [34]:
## change the config to use a different session and see the difference in response
config1={"configurable":{"session_id": "chat3"}}
response1 = with_message_history.invoke([HumanMessage(content="what is my name and profession?")], 
                            config=config1)
response1.content


"I still don't have any information about your name or profession. As I mentioned earlier, I'm a large language model, I don't have personal data about users, and our conversation just started. If you'd like to share your name and profession, I'd be happy to chat with you about it!"

In [33]:
print(store)

{'chat1': InMemoryChatMessageHistory(messages=[HumanMessage(content='hey! im prachi and im aiml engineer, passionate about learinging new things, can you tell me about langchain? ', additional_kwargs={}, response_metadata={}), AIMessage(content="Nice to meet you, Prachi. As an AIML engineer, you're likely interested in staying up-to-date with the latest developments in the field. LangChain is a relatively new and exciting concept that has gained significant attention in the AI community.\n\nLangChain is an open-source framework designed to help developers build applications that utilize large language models (LLMs) more efficiently. It was created to simplify the process of integrating LLMs into various projects, making it easier to harness their power.\n\nThe primary goal of LangChain is to provide a set of tools and APIs that allow developers to focus on building their applications, rather than worrying about the underlying complexities of LLMs. By abstracting away the low-level deta

## Prompt template

In [54]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that helps answer questions about the user in {language}."),
    MessagesPlaceholder(variable_name="messages")
])

chain=prompt|model

In [45]:
chain.invoke({"messages": [HumanMessage(content="hey! im prachi and im aiml engineer. ")]})

AIMessage(content="Hello Prachi, nice to meet you. That's really cool that you're an AIML (Artificial Intelligence and Machine Learning) engineer. What specific areas of AIML interest you the most, or what kind of projects have you been working on recently?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 60, 'total_tokens': 113, 'completion_time': 0.133769276, 'completion_tokens_details': None, 'prompt_time': 0.002127306, 'prompt_tokens_details': None, 'queue_time': 0.163767653, 'total_time': 0.135896582}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dfa02-dd9e-7473-8d3a-5d568aa32d6d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 53, 'total_tokens': 113})

In [46]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)


In [47]:
config = {"configurable":{"session_id": "chat4"}}
response = with_message_history.invoke([HumanMessage(content="hey! im prachi and im aiml engineer")], 
                            config=config)  

In [48]:
response

AIMessage(content="Hello Prachi, nice to meet you. That's really cool that you're an AIML (Artificial Intelligence and Machine Learning) engineer. What kind of projects have you been working on, or what areas of AIML interest you the most?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 58, 'total_tokens': 109, 'completion_time': 0.153727868, 'completion_tokens_details': None, 'prompt_time': 0.002958927, 'prompt_tokens_details': None, 'queue_time': 0.053833773, 'total_time': 0.156686795}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dfa04-125e-7203-802a-8111d2b85837-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 51, 'total_tokens': 109})

In [55]:
response2= chain.invoke({"messages": [HumanMessage(content="hey! im prachi and im aiml engineer. ")], "language": "hindi"})

In [56]:
response2.content

'नमस्ते प्राची! आप एक एआईएमएल इंजीनियर हैं, यह बहुत अच्छा है। आपका यह क्षेत्र बहुत रोचक है और भविष्य में इसकी बहुत संभावनाएं हैं। आपकी रुचि और अनुभव क्या हैं? आप किस प्रकार के परियोजनाओं पर काम करती हैं?'

## managing the conversation history 

* it's imp concept to understand when building chatbots it's imp to manage conversation history 
* if left unmannaged , the list ofmessages will grow unbounded and potentially overflow the contetx window llm
* it's imp to add a step that limits the size of the messages u r passing in 
* 'trim_messages' helper to reduse how many messages we're sending to the model
* it basically allows how many tokens we want to keep 

In [59]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(max_tokens=70,
                        strategy="last",
                        token_counter=model,
                        include_system=True,
                        allow_partial=True,
                        start_on="human")

In [61]:
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="hey! i m bob the builder "),
    AIMessage(content="hi! how can i help you?"),
    HumanMessage(content="i want to watch a movie"),
    AIMessage(content="sure! which one?"),
    HumanMessage(content="i want to watch spiderman"),
    AIMessage(content="great choice! which spiderman movie do you want to watch?"),
    HumanMessage(content="what is 2+2? "),
    AIMessage(content="2+2 equals 4.") ,
    HumanMessage(content="what is the capital of india? "),
    AIMessage(content="the capital of india is new delhi")  
]

In [62]:
trimmer.invoke(messages)

c:\Users\prakr\OneDrive\local_dekstop\langchain\.venv\Lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
c:\Users\prakr\OneDrive\local_dekstop\langchain\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prakr\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate De

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='i want to watch spiderman', additional_kwargs={}, response_metadata={}),
 AIMessage(content='great choice! which spiderman movie do you want to watch?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what is 2+2? ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='2+2 equals 4.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what is the capital of india? ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='the capital of india is new delhi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [63]:
## in the output the top 2 messages are removed ^

In [76]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer) | prompt| model)

In [78]:
chain.invoke(
    {"messages": messages + [HumanMessage(content="what is the capital of india? ")],
     "language": "english"}
)

AIMessage(content='The capital of India is New Delhi.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 115, 'total_tokens': 124, 'completion_time': 0.019403071, 'completion_tokens_details': None, 'prompt_time': 0.00493143, 'prompt_tokens_details': None, 'queue_time': 0.05448976, 'total_time': 0.024334501}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dfa1b-1ae2-7910-b5a8-da5fd4d9d24c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 115, 'output_tokens': 9, 'total_tokens': 124})